In [2]:
r"""
=================================================================================
ORALVISION-M4 — STATISTICAL RIGOR SUITE (Bootstrap CIs + McNemar's + Paired Diffs)
Closes out Supervisor Review Item #6 in a single run.
=================================================================================

WHAT THIS SCRIPT DOES
----------------------
1. Loads your already-trained, clean-dataset-retrained OralVision-M4 checkpoint
   (best_model.pth) — no retraining, this is pure inference + statistics.
2. Runs inference on the internal TEST set -> saves per-image true label,
   predicted label, and full softmax probability vector to CSV
   (test_predictions_M4_joint_head.csv).
3. Extracts frozen embeddings from the same backbone and fits the four
   post-hoc classifier families (Logistic Regression, SVM-RBF, Random Forest,
   XGBoost) -> saves per-image predictions + probabilities for each, on the
   SAME 1,265 test images (required for any paired statistical comparison).
4. Computes bootstrap 95% confidence intervals (10,000 resamples, seed=42) for:
     - Accuracy, Macro-Precision, Macro-Recall, Macro-F1  (all five models)
     - Per-class recall (sensitivity), all six classes    (all five models)
     - Macro-averaged one-vs-rest ROC-AUC                 (all five models)
5. Computes PAIRED comparisons between the M4 joint head and each of the four
   post-hoc classifiers, all evaluated on the identical test images:
     - McNemar's test (continuity-corrected)
     - Paired bootstrap 95% CI for the accuracy difference
     - Paired bootstrap 95% CI for the macro-F1 difference
     - Paired bootstrap 95% CI for the macro-AUC difference
6. Writes one human-readable report (.txt) and one machine-readable (.json),
   formatted so the numbers drop straight into the manuscript's stats section.

>>> EDIT CHECKPOINT_PATH AND DATASET_ROOT BELOW, THEN RUN. <<<
=================================================================================
"""

import os
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import label_binarize
from scipy.stats import chi2

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


# =============================================================================
# CONFIG — EDIT THESE TWO FOR YOUR MACHINE
# =============================================================================

CHECKPOINT_PATH = r"C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth"
DATASET_ROOT = Path(r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN")
OUTPUT_DIR = r"C:\Users\R Y Z E N\Desktop\m4_statistical_rigor_report"

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
N_BOOTSTRAP = 10000  # drop to 2000 for a quick sanity check, 10000 for the paper

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


# =============================================================================
# MODEL DEFINITION — must match the checkpoint's architecture exactly
# (identical to the classifier-family-comparison script you already ran)
# =============================================================================
class CoordAtt(nn.Module):
    def __init__(self, channels, reduction=32):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, channels // reduction)
        self.conv1 = nn.Conv2d(channels, mip, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)
        self.conv_h = nn.Conv2d(mip, channels, kernel_size=1)
        self.conv_w = nn.Conv2d(mip, channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        _, _, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.sigmoid(self.conv_h(x_h))
        a_w = self.sigmoid(self.conv_w(x_w))
        return identity * a_h * a_w


class EfficientNetCoordAtt(nn.Module):
    def __init__(self, num_classes, ca_reduction=32):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.coordatt = CoordAtt(1280, reduction=ca_reduction)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, num_classes)

    def embed(self, x):
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        return self.classifier(self.embed(x))


# =============================================================================
# DATASET — same recursive-search discovery as the original training script,
# so this finds train/val/test wherever they actually sit under DATASET_ROOT
# instead of assuming they're immediate children of it.
# =============================================================================
def find_split_directory(root, names):
    """Recursively find the train/val/test directory (>=2 subfolders = classes)."""
    if isinstance(names, str):
        names = [names]
    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        if p.name.lower() in {x.lower() for x in names}:
            class_dirs = [x for x in p.iterdir() if x.is_dir()]
            if len(class_dirs) >= 2:
                candidates.append((len(class_dirs), p))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset directory not found:\n{DATASET_ROOT}")

TRAIN_DIR = find_split_directory(DATASET_ROOT, "train")
TEST_DIR = find_split_directory(DATASET_ROOT, "test")

if TRAIN_DIR is None:
    raise FileNotFoundError(f"Could not find a TRAIN directory under:\n{DATASET_ROOT}")
if TEST_DIR is None:
    raise FileNotFoundError(f"Could not find a TEST directory under:\n{DATASET_ROOT}")


def discover_classes(train_dir, test_dir):
    train_classes = {p.name for p in train_dir.iterdir() if p.is_dir()}
    test_classes = {p.name for p in test_dir.iterdir() if p.is_dir()}
    if train_classes != test_classes:
        raise RuntimeError(
            f"Class mismatch between train/test.\nTrain: {sorted(train_classes)}\nTest: {sorted(test_classes)}"
        )
    return sorted(train_classes)


CLASS_NAMES = discover_classes(TRAIN_DIR, TEST_DIR)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)


def collect_samples(split_dir):
    samples = []
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                samples.append((str(path), CLASS_TO_IDX[class_name]))
    return samples


class FolderDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            img = img.convert("RGB")
        return self.transform(img), label


eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# =============================================================================
# INFERENCE HELPERS
# =============================================================================
@torch.no_grad()
def run_joint_head_inference(model, loader):
    model.eval()
    probs, labels = [], []
    for imgs, y in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        p = torch.softmax(logits, dim=1).cpu().numpy()
        probs.append(p)
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)


@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval()
    feats, labels = [], []
    for imgs, y in loader:
        imgs = imgs.to(DEVICE)
        emb = model.embed(imgs)
        feats.append(emb.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


# =============================================================================
# METRIC HELPERS
# =============================================================================
def per_class_and_macro(y_true, y_pred):
    """One sklearn call -> per-class P/R/F1 + macro-averages + accuracy."""
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0, labels=range(NUM_CLASSES)
    )
    acc = accuracy_score(y_true, y_pred)
    return acc, p.mean(), r.mean(), f1.mean(), r  # r = per-class recall array


def macro_auc(y_true, y_prob):
    y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    try:
        return roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr")
    except ValueError:
        return np.nan


def ci(values):
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[~np.isnan(vals)]
    return {
        "mean": float(np.mean(vals)),
        "ci_lower": float(np.percentile(vals, 2.5)),
        "ci_upper": float(np.percentile(vals, 97.5)),
    }


def bootstrap_single_model(y_true, y_pred, y_prob, n_boot=N_BOOTSTRAP, tag=""):
    n = len(y_true)
    rng = np.random.default_rng(SEED)
    idx_all = np.arange(n)

    accs, precs, recs, f1s, aucs = [], [], [], [], []
    class_recalls = np.zeros((n_boot, NUM_CLASSES))

    for b in range(n_boot):
        idx = rng.choice(idx_all, size=n, replace=True)
        yt, yp = y_true[idx], y_pred[idx]
        acc, mp, mr, mf1, per_class_r = per_class_and_macro(yt, yp)
        accs.append(acc); precs.append(mp); recs.append(mr); f1s.append(mf1)
        class_recalls[b] = per_class_r
        if y_prob is not None:
            aucs.append(macro_auc(yt, y_prob[idx]))

    result = {
        "accuracy": ci(accs),
        "macro_precision": ci(precs),
        "macro_recall": ci(recs),
        "macro_f1": ci(f1s),
    }
    if y_prob is not None:
        result["macro_auc"] = ci(aucs)
    result["per_class_recall"] = {
        CLASS_NAMES[c]: ci(class_recalls[:, c]) for c in range(NUM_CLASSES)
    }
    log(f"  [{tag}] bootstrap done -> Acc {result['accuracy']['mean']:.4f} "
        f"[{result['accuracy']['ci_lower']:.4f}, {result['accuracy']['ci_upper']:.4f}], "
        f"F1 {result['macro_f1']['mean']:.4f} "
        f"[{result['macro_f1']['ci_lower']:.4f}, {result['macro_f1']['ci_upper']:.4f}]")
    return result


def mcnemar_test(y_true, pred_a, pred_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))   # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))   # a wrong, b right
    if b + c == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0}
    stat = (abs(b - c) - 1) ** 2 / (b + c)     # continuity-corrected
    p = float(1 - chi2.cdf(stat, df=1))
    return {"b": b, "c": c, "statistic": float(stat), "p_value": p}


def paired_bootstrap_diff(y_true, pred_a, pred_b, prob_a=None, prob_b=None, n_boot=N_BOOTSTRAP):
    n = len(y_true)
    rng = np.random.default_rng(SEED)
    idx_all = np.arange(n)

    acc_diffs, f1_diffs, auc_diffs = [], [], []
    for b in range(n_boot):
        idx = rng.choice(idx_all, size=n, replace=True)
        yt = y_true[idx]

        acc_a, _, _, f1_a, _ = per_class_and_macro(yt, pred_a[idx])
        acc_b, _, _, f1_b, _ = per_class_and_macro(yt, pred_b[idx])
        acc_diffs.append(acc_a - acc_b)
        f1_diffs.append(f1_a - f1_b)

        if prob_a is not None and prob_b is not None:
            auc_a = macro_auc(yt, prob_a[idx])
            auc_b = macro_auc(yt, prob_b[idx])
            auc_diffs.append(auc_a - auc_b)

    result = {
        "accuracy_diff": ci(acc_diffs),
        "macro_f1_diff": ci(f1_diffs),
    }
    if auc_diffs:
        result["macro_auc_diff"] = ci(auc_diffs)
    return result


# =============================================================================
# MAIN
# =============================================================================
def main():
    t_start = time.time()
    log(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        log(f"GPU: {torch.cuda.get_device_name(0)}")

    log(f"Train directory: {TRAIN_DIR}")
    log(f"Test directory : {TEST_DIR}")
    log(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

    TRAIN_SAMPLES = collect_samples(TRAIN_DIR)
    TEST_SAMPLES = collect_samples(TEST_DIR)
    if not TRAIN_SAMPLES or not TEST_SAMPLES:
        raise RuntimeError("One or both dataset splits contain zero images — check DATASET_ROOT.")

    train_ds = FolderDataset(TRAIN_SAMPLES, eval_transform)
    test_ds = FolderDataset(TEST_SAMPLES, eval_transform)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    log(f"Train: {len(train_ds)} images | Test: {len(test_ds)} images")

    # ---- Load checkpoint ----
    log(f"Loading checkpoint: {CHECKPOINT_PATH}")
    model = EfficientNetCoordAtt(num_classes=NUM_CLASSES).to(DEVICE)
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing or unexpected:
        log(f"  [WARN] missing={missing} unexpected={unexpected} — verify architecture before trusting results.")
    else:
        log("  Checkpoint loaded cleanly.")

    # ---- Step 1: joint-head (M4) inference on test set ----
    log("Running M4 joint-head inference on test set...")
    prob_m4, y_test = run_joint_head_inference(model, test_loader)
    pred_m4 = np.argmax(prob_m4, axis=1)

    # ---- Step 2: frozen embeddings + post-hoc classifiers ----
    log("Extracting frozen embeddings (train + test)...")
    X_train, y_train = extract_embeddings(model, train_loader)
    X_test, y_test_emb = extract_embeddings(model, test_loader)
    assert np.array_equal(y_test, y_test_emb), "Test label ordering mismatch between loaders!"

    classifiers = {
        "LogReg": LogisticRegression(max_iter=2000, random_state=SEED),
        "SVM_RBF": SVC(kernel="rbf", probability=True, random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    }
    if HAS_XGB:
        classifiers["XGBoost"] = XGBClassifier(
            n_estimators=300, use_label_encoder=False, eval_metric="mlogloss",
            random_state=SEED, n_jobs=-1,
        )
    else:
        log("  [WARN] xgboost not installed — skipping. pip install xgboost --break-system-packages")

    predictions = {"M4_joint_head": (pred_m4, prob_m4)}
    for name, clf in classifiers.items():
        log(f"Fitting {name}...")
        t0 = time.time()
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        prob = clf.predict_proba(X_test) if hasattr(clf, "predict_proba") else None
        predictions[name] = (pred, prob)
        log(f"  {name} done in {time.time()-t0:.1f}s")

    # ---- Save all per-image predictions to CSV ----
    import csv
    pred_csv = os.path.join(OUTPUT_DIR, "test_predictions_all_models.csv")
    with open(pred_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        header = ["image_index", "true_label"] + [f"{name}_pred" for name in predictions]
        writer.writerow(header)
        for i in range(len(y_test)):
            row = [i, CLASS_NAMES[y_test[i]]] + [CLASS_NAMES[predictions[name][0][i]] for name in predictions]
            writer.writerow(row)
    log(f"Wrote {pred_csv}")

    # ---- Step 3: bootstrap CIs for every model ----
    log(f"Running bootstrap CIs ({N_BOOTSTRAP} resamples per model)...")
    bootstrap_results = {}
    for name, (pred, prob) in predictions.items():
        bootstrap_results[name] = bootstrap_single_model(y_test, pred, prob, tag=name)

    # ---- Step 4: paired comparisons (M4 vs each classifier) ----
    log("Running paired comparisons (McNemar's + paired bootstrap diffs)...")
    paired_results = {}
    m4_pred, m4_prob = predictions["M4_joint_head"]
    for name, (pred, prob) in predictions.items():
        if name == "M4_joint_head":
            continue
        log(f"  M4 vs {name} ...")
        mcnemar = mcnemar_test(y_test, m4_pred, pred)
        diffs = paired_bootstrap_diff(y_test, m4_pred, pred, m4_prob, prob)
        paired_results[f"M4_vs_{name}"] = {"mcnemar": mcnemar, **diffs}

    # ---- Write final report ----
    full_report = {
        "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
        "n_bootstrap": N_BOOTSTRAP,
        "n_test_images": int(len(y_test)),
        "bootstrap_ci": bootstrap_results,
        "paired_comparisons": paired_results,
    }
    json_path = os.path.join(OUTPUT_DIR, "statistical_rigor_report.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(full_report, f, indent=2)

    txt_path = os.path.join(OUTPUT_DIR, "statistical_rigor_report.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("OralVision-M4 — Statistical Rigor Report\n")
        f.write("=" * 70 + "\n")
        f.write(f"Generated: {full_report['generated']}\n")
        f.write(f"Bootstrap resamples: {N_BOOTSTRAP}\n")
        f.write(f"Test images: {full_report['n_test_images']}\n\n")

        f.write("BOOTSTRAP 95% CONFIDENCE INTERVALS\n")
        f.write("-" * 70 + "\n")
        for name, res in bootstrap_results.items():
            f.write(f"\n{name}\n")
            for metric in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_auc"]:
                if metric in res:
                    m = res[metric]
                    f.write(f"  {metric:18s}: {m['mean']:.4f}  [{m['ci_lower']:.4f}, {m['ci_upper']:.4f}]\n")
            f.write("  per-class recall:\n")
            for cname, m in res["per_class_recall"].items():
                f.write(f"    {cname:22s}: {m['mean']:.4f}  [{m['ci_lower']:.4f}, {m['ci_upper']:.4f}]\n")

        f.write("\n\nPAIRED COMPARISONS (M4 joint head vs. each post-hoc classifier)\n")
        f.write("-" * 70 + "\n")
        for pair, res in paired_results.items():
            f.write(f"\n{pair}\n")
            mc = res["mcnemar"]
            f.write(f"  McNemar's test: b={mc['b']}, c={mc['c']}, "
                    f"chi2={mc['statistic']:.4f}, p={mc['p_value']:.4g}\n")
            for metric in ["accuracy_diff", "macro_f1_diff", "macro_auc_diff"]:
                if metric in res:
                    m = res[metric]
                    f.write(f"  {metric:18s}: {m['mean']:+.4f}  [{m['ci_lower']:+.4f}, {m['ci_upper']:+.4f}]\n")

    log(f"Wrote {json_path}")
    log(f"Wrote {txt_path}")
    log(f"TOTAL RUNTIME: {time.time() - t_start:.1f}s")


if __name__ == "__main__":
    main()

[06:03:47] Device: cuda
[06:03:47] GPU: NVIDIA GeForce RTX 2050
[06:03:47] Train directory: C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\train
[06:03:47] Test directory : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\test
[06:03:47] Classes (6): ['Calculus', 'Caries', 'Gingivitis', 'Hypodontia', 'Mouth_Ulcer', 'Tooth_Discoloration']
[06:03:49] Train: 17700 images | Test: 1265 images
[06:03:49] Loading checkpoint: C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED\best_model.pth
[06:03:49]   Checkpoint loaded cleanly.
[06:03:49] Running M4 joint-head inference on test set...
[06:04:00] Extracting frozen embeddings (train + test)...
[06:05:59] Fitting LogReg...
[06:06:00]   LogReg done in 1.2s
[06:06:00] Fitting SVM_RBF...
[06:06:19]   SVM_RBF done in 18.9s
[06:06:19] Fitting RandomForest...
[06:06:31]   RandomForest done in 11.6s
[06:06:31] Fitting XGBoost...


C:\Users\R Y Z E N\shopgenie\lib\site-packages\xgboost\training.py:183: UserWarning: [06:06:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[06:08:06]   XGBoost done in 95.1s
[06:08:06] Wrote C:\Users\R Y Z E N\Desktop\m4_statistical_rigor_report\test_predictions_all_models.csv
[06:08:06] Running bootstrap CIs (10000 resamples per model)...
[06:09:17]   [M4_joint_head] bootstrap done -> Acc 0.9953 [0.9913, 0.9984], F1 0.9840 [0.9667, 0.9974]
[06:10:22]   [LogReg] bootstrap done -> Acc 0.9937 [0.9889, 0.9976], F1 0.9768 [0.9566, 0.9925]
[06:11:23]   [SVM_RBF] bootstrap done -> Acc 0.9905 [0.9850, 0.9953], F1 0.9658 [0.9425, 0.9848]
[06:12:21]   [RandomForest] bootstrap done -> Acc 0.9921 [0.9874, 0.9961], F1 0.9716 [0.9500, 0.9887]
[06:13:23]   [XGBoost] bootstrap done -> Acc 0.9889 [0.9826, 0.9945], F1 0.9557 [0.9277, 0.9780]
[06:13:23] Running paired comparisons (McNemar's + paired bootstrap diffs)...
[06:13:23]   M4 vs LogReg ...
[06:15:34]   M4 vs SVM_RBF ...
[06:17:46]   M4 vs RandomForest ...
[06:19:49]   M4 vs XGBoost ...
[06:21:52] Wrote C:\Users\R Y Z E N\Desktop\m4_statistical_rigor_report\statistical_rigor_report